# lensless reconstruction demo

paste a google drive zip link in the config cell below, run all, reconstructions land in `OUTPUT_DIR`.
i run my best model here — leaddm5 + drunet pre+post (~17.3 db on digicam test).

zip should look like this:

```
dataset/
  lensless/
  masks/
  lensed/   # optional
```

`lensed/` is only if you want gt plots + metrics. without it inference and viz still work, i just skip metrics.

## setup

i clone my repo, install what i put in `requirements.txt`, and pull in `gdown` for drive zips. same stack as the README, minus the venv stuff colab doesn't need. safe to re-run — clone is skipped if the folder's already there.

In [ ]:
import os

REPO_URL = "https://github.com/tolyaho/lensless-computational-imaging.git"
REPO_DIR = "/content/lensless-computational-imaging"

if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt
!pip install -q gdown

## input

put your google drive zip link in the cell below. by default i save reconstructions to `/content/lensless_demo_outputs/reconstructions`.

In [ ]:
DATASET_ZIP_URL = "PASTE_GOOGLE_DRIVE_ZIP_URL_HERE"

UNZIP_DIR = "/content/lensless_demo_data"
OUTPUT_DIR = "/content/lensless_demo_outputs/reconstructions"

MODEL_PRESET = "leadmm5_drunet8m_prepost"
NUM_VIS_SAMPLES = 5

print("output dir:", OUTPUT_DIR)

## checkpoints

i pull the final model weights with `download_checkpoints.py` — no manual upload. demo defaults to my pre+post drunet checkpoint; the script grabs all report checkpoints in one go.

In [ ]:
!python download_checkpoints.py --all

## data

downloads your zip from drive (or a direct url), unpacks into `UNZIP_DIR`, then i hunt for the folder that actually has `lensless/` and `masks/` inside — the zip might nest it one level deep.

In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path

import gdown


def find_dataset_root(base_dir):
    base_dir = Path(base_dir)
    candidates = [base_dir] + [p for p in base_dir.rglob("*") if p.is_dir()]
    for p in candidates:
        if (p / "lensless").is_dir() and (p / "masks").is_dir():
            return p
    raise FileNotFoundError("could not find dataset root with lensless/ and masks/")


def download_zip(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if "drive.google.com" in url or "docs.google.com" in url:
        gdown.download(url, str(dest), fuzzy=True)
        return

    try:
        import requests

        response = requests.get(url, timeout=120)
        response.raise_for_status()
        dest.write_bytes(response.content)
    except Exception:
        if shutil.which("wget"):
            subprocess.run(["wget", "-O", str(dest), url], check=True)
        else:
            raise


if "PASTE_GOOGLE_DRIVE" in DATASET_ZIP_URL:
    raise ValueError("set DATASET_ZIP_URL in the input cell above")

zip_path = Path(f"{UNZIP_DIR}.zip")
unzip_dir = Path(UNZIP_DIR)
unzip_dir.mkdir(parents=True, exist_ok=True)

download_zip(DATASET_ZIP_URL, zip_path)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(unzip_dir)

DATASET_ROOT = find_dataset_root(unzip_dir)

lensless_dir = DATASET_ROOT / "lensless"
masks_dir = DATASET_ROOT / "masks"
lensed_dir = DATASET_ROOT / "lensed"

print("dataset root:", DATASET_ROOT)
print("lensless files:", len(list(lensless_dir.iterdir())))
print("masks files:", len(list(masks_dir.iterdir())))
print("lensed/ exists:", lensed_dir.is_dir())

## inference

i run my best report model on your custom folder through `inference.py` — no copied inference loop in the notebook. reconstructions land in `OUTPUT_DIR/recon`, and if the zip had `lensed/` i also save aligned gt to `OUTPUT_DIR/lensed`.

In [ ]:
CHECKPOINT_PATH = "checkpoints/leadmm5_prepost_drunet8m.pth"

!python inference.py -cn inference_custom \
    model=modular_leadmm5_prepost_drunet8m \
    checkpoint={CHECKPOINT_PATH} \
    dataset.root={DATASET_ROOT} \
    output_dir={OUTPUT_DIR} \
    save_targets=true

RECON_DIR = Path(OUTPUT_DIR) / "recon"
print("reconstructions saved to:", RECON_DIR)

## samples

quick side-by-side: original (if your zip had `lensed/`), lensless measurement, my reconstruction. i match files by stem so `.png` vs `.jpg` doesn't trip it up.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

IMAGE_EXTS = (".png", ".jpg", ".jpeg")


def index_images(folder: Path) -> dict[str, Path]:
    images = {}
    if not folder.is_dir():
        return images
    for path in sorted(folder.iterdir()):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
            images[path.stem] = path
    return images


def find_by_stem(folder: Path, stem: str) -> Path | None:
    if not folder.is_dir():
        return None
    for ext in IMAGE_EXTS:
        candidate = folder / f"{stem}{ext}"
        if candidate.is_file():
            return candidate
    return None


recon_dir = Path(OUTPUT_DIR) / "recon"
lensless_dir = DATASET_ROOT / "lensless"
lensed_dir = DATASET_ROOT / "lensed"

recon_files = index_images(recon_dir)
if not recon_files:
    raise FileNotFoundError(f"no reconstructions found in {recon_dir}")

sample_ids = sorted(recon_files.keys())[:NUM_VIS_SAMPLES]
show_original = lensed_dir.is_dir()

for stem in sample_ids:
    panels = []
    titles = []

    if show_original:
        lensed_path = find_by_stem(lensed_dir, stem)
        if lensed_path is not None:
            panels.append(Image.open(lensed_path).convert("RGB"))
            titles.append("original")
        else:
            panels.append(Image.new("RGB", Image.open(recon_files[stem]).size, color=(32, 32, 32)))
            titles.append("original (missing)")

    lensless_path = find_by_stem(lensless_dir, stem)
    if lensless_path is None:
        raise FileNotFoundError(f"no lensless image for stem {stem!r} in {lensless_dir}")

    panels.append(Image.open(lensless_path).convert("RGB"))
    titles.append("lensless")

    panels.append(Image.open(recon_files[stem]).convert("RGB"))
    titles.append("reconstruction")

    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
    if len(panels) == 1:
        axes = [axes]

    for ax, img, title in zip(axes, panels, titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")

    fig.suptitle(stem)
    plt.show()

print(f"showing {len(sample_ids)} / {len(recon_files)} reconstructions")

## metrics

if your zip had gt in `lensed/`, i run the same `calculate_metrics.py` as the report (roi crop on, same four metrics). no gt → i skip this cleanly.

In [ ]:
import json


def has_gt_images(folder: Path) -> bool:
    if not folder.is_dir():
        return False
    for ext in IMAGE_EXTS:
        if any(folder.glob(f"*{ext}")):
            return True
    return False


gt_dir = None
for candidate in (Path(OUTPUT_DIR) / "lensed", DATASET_ROOT / "lensed"):
    if has_gt_images(candidate):
        gt_dir = candidate
        break

pred_dir = Path(OUTPUT_DIR) / "recon"
metrics_path = Path(OUTPUT_DIR) / "metrics.json"

if gt_dir is None:
    print("no lensed ground truth in dataset or output — skipping metrics")
else:
    !python calculate_metrics.py pred_dir={pred_dir} gt_dir={gt_dir} output_path={metrics_path} use_roi=true

    results = json.loads(metrics_path.read_text())
    for name in ("PSNR", "SSIM", "MSE", "LPIPS"):
        print(f"{name}: {results[name]}")

## done

reconstructions are in `OUTPUT_DIR`. if your dataset had gt, metrics are printed above too.

In [ ]:
recon_dir = Path(OUTPUT_DIR) / "recon"
metrics_path = Path(OUTPUT_DIR) / "metrics.json"

print("dataset root:", DATASET_ROOT)
print("reconstruction folder:", recon_dir)
print("metrics path:", metrics_path if metrics_path.is_file() else "(not computed)")

In [ ]:
!zip -qr reconstructions.zip {recon_dir}
print("saved reconstructions.zip")